In [1]:
from data_frame.analytical.window_function.window_definer import WindowDefiner
from data_frame.spark_utils import get_spark
from pyspark.sql import functions as F
from data_frame.analytical.window_function.rank_calculator import RankCalculator

In [2]:
spark = get_spark(app_name="Ranking functions")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 12:15:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Create test data
scores_data = [
    ("Math", "Alice", 95),
    ("Math", "Bob", 85),
    ("Math", "Charlie", 95),
    ("Math", "David", 85),
    ("Math", "Eve", 75),
    ("Science", "Alice", 90),
    ("Science", "Bob", 85),
    ("Science", "Charlie", 95),
    ("Science", "David", 85)
]
df_scores = spark.createDataFrame(scores_data, ["subject", "student", "score"])

## 1. Different Ranking Types

In [4]:
# Define window
window_spec = WindowDefiner.define_ordered_window(
    partition_cols=["subject"],
    order_cols=["score"],
    order_direction="desc"
)

# Apply different ranking functions
df_ranked = RankCalculator.rank_within_partition(df_scores, window_spec, "rank")
df_ranked = RankCalculator.dense_rank_within_partition(df_ranked, window_spec, "dense_rank")
df_ranked = RankCalculator.row_number_within_partition(df_ranked, window_spec, "row_num")
df_ranked = RankCalculator.percent_rank_within_partition(df_ranked, window_spec, "pct_rank")

print("Comparison of ranking functions:")
df_ranked.orderBy("subject", "score", "student").show()

Comparison of ranking functions:


+-------+-------+-----+----+----------+-------+------------------+
|subject|student|score|rank|dense_rank|row_num|          pct_rank|
+-------+-------+-----+----+----------+-------+------------------+
|   Math|    Eve|   75|   5|         3|      5|               1.0|
|   Math|    Bob|   85|   3|         2|      3|               0.5|
|   Math|  David|   85|   3|         2|      4|               0.5|
|   Math|  Alice|   95|   1|         1|      1|               0.0|
|   Math|Charlie|   95|   1|         1|      2|               0.0|
|Science|    Bob|   85|   3|         3|      3|0.6666666666666666|
|Science|  David|   85|   3|         3|      4|0.6666666666666666|
|Science|  Alice|   90|   2|         2|      2|0.3333333333333333|
|Science|Charlie|   95|   1|         1|      1|               0.0|
+-------+-------+-----+----+----------+-------+------------------+



## 2. NTILE (Bucket) Function

In [7]:
# Define window for NTILE
window_ntile = WindowDefiner.define_ordered_window(
    partition_cols=["subject"],
    order_cols=["score"],
    order_direction="desc"
)

# Calculate NTILE (quartiles)
df_ntiled = RankCalculator.ntile_within_partition(
    df_scores, window_ntile, 4, "quartile"
)

print("NTILE (quartiles) by subject:")
df_ntiled.orderBy(["subject", "score"], ascending=[True, False]).show()

NTILE (quartiles) by subject:
+-------+-------+-----+--------+
|subject|student|score|quartile|
+-------+-------+-----+--------+
|   Math|  Alice|   95|       1|
|   Math|Charlie|   95|       1|
|   Math|    Bob|   85|       2|
|   Math|  David|   85|       3|
|   Math|    Eve|   75|       4|
|Science|Charlie|   95|       1|
|Science|  Alice|   90|       2|
|Science|    Bob|   85|       3|
|Science|  David|   85|       4|
+-------+-------+-----+--------+



## 3. Top N per Group

In [8]:
# Get top 2 scores per subject
window_top = WindowDefiner.define_ordered_window(
    partition_cols=["subject"],
    order_cols=["score"],
    order_direction="desc"
)

df_with_row_num = RankCalculator.row_number_within_partition(
    df_scores, window_top, "row_num"
)

top_2_per_subject = df_with_row_num.filter(F.col("row_num") <= 2)
print("Top 2 scores per subject:")
top_2_per_subject.orderBy("subject", "row_num").show()

Top 2 scores per subject:
+-------+-------+-----+-------+
|subject|student|score|row_num|
+-------+-------+-----+-------+
|   Math|  Alice|   95|      1|
|   Math|Charlie|   95|      2|
|Science|Charlie|   95|      1|
|Science|  Alice|   90|      2|
+-------+-------+-----+-------+

